## 전처리

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import sys
from scipy.stats import spearmanr

In [2]:
# 1. scan_csv를 사용하여 지연 평가(Lazy) 방식으로 연결해
# 인코딩이 CP949일 가능성이 높으니 에러가 나면 encoding="cp949"를 추가해줘
try:
    # 데이터 경로를 실제 파일명으로 수정해!
    pos_lazy = pl.scan_csv("/Users/hajiyoon/dataset_seveneleven/B2_POS_SALE_H1.csv") 
    
    # 2. head(10)을 호출하고 collect()로 실행해
    # 지연 평가 방식이라 collect()를 해야 실제 결과가 출력돼
    pos_head = pos_lazy.head(10).collect()
    
    print("--- POS 데이터 상위 10개 행 ---")
    print(pos_head)

except Exception as e:
    print(f"파일을 읽는 중 에러가 발생했어: {e}")
    print("팁: 인코딩 문제라면 scan_csv(..., encoding='cp949')를 시도해봐!")

--- POS 데이터 상위 10개 행 ---
shape: (10, 8)
┌───────────┬───────────┬────────────┬────────┬──────────┬───────────┬──────────┬──────────┐
│ SALE_DATE ┆ SALE_TIME ┆ STORE_CODE ┆ POS_NO ┆ TRADE_NO ┆ ITEM_CODE ┆ SALE_QTY ┆ SALE_AMT │
│ ---       ┆ ---       ┆ ---        ┆ ---    ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ i64       ┆ i64       ┆ i64        ┆ i64    ┆ i64      ┆ i64       ┆ i64      ┆ str      │
╞═══════════╪═══════════╪════════════╪════════╪══════════╪═══════════╪══════════╪══════════╡
│ 20250523  ┆ 84531     ┆ 64139      ┆ 2      ┆ 26651    ┆ 314952    ┆ 1        ┆ 500      │
│ 20250523  ┆ 105724    ┆ 52657      ┆ 1      ┆ 48727    ┆ 314954    ┆ 10       ┆ 45,000   │
│ 20250523  ┆ 105724    ┆ 52657      ┆ 1      ┆ 48727    ┆ 314952    ┆ 10       ┆ 5,000    │
│ 20250523  ┆ 84531     ┆ 64139      ┆ 2      ┆ 26651    ┆ 314983    ┆ 1        ┆ 4,500    │
│ 20250523  ┆ 83823     ┆ 50134      ┆ 1      ┆ 67971    ┆ 314988    ┆ 1        ┆ 4,500    │
│ 20250523  ┆ 113903    ┆ 6807

In [ ]:
# 1. 스캔 단계: 쉼표가 들어있을 수 있는 수량/금액 컬럼을 무조건 문자열(String)로 읽도록 강제함
# (구버전 Polars의 경우 pl.String 대신 pl.Utf8을 사용해)
pos_lazy = pl.scan_csv(
    "/Users/hajiyoon/dataset_seveneleven/B2_POS_SALE_H1.csv", 
    schema_overrides={
        "SALE_QTY": pl.String, 
        "SALE_AMT": pl.String
    }
)

# 2. 전처리 파이프라인: 수량(SALE_QTY)에도 쉼표 제거 로직 추가!
pos_processed_lazy = pos_lazy.select([
    pl.col("SALE_DATE").cast(pl.String).alias("판매일자"),
    pl.col("SALE_TIME").cast(pl.String).str.zfill(6).alias("판매시간"),
    pl.col("STORE_CODE").cast(pl.String).alias("점포코드"),
    pl.col("POS_NO").cast(pl.String).str.zfill(2).alias("POS번호"),
    pl.col("TRADE_NO").cast(pl.String).alias("거래번호"),
    pl.col("ITEM_CODE").cast(pl.String).alias("상품코드"),
    
    # 🌟 추가된 부분: 수량에서도 쉼표(,)를 제거하고 정수로 변환해!
    pl.col("SALE_QTY").str.replace_all(",", "").cast(pl.Int64).alias("판매수량"),
    
    # 금액은 아까처럼 쉼표 제거 후 정수로 변환
    pl.col("SALE_AMT").str.replace_all(",", "").cast(pl.Int64).alias("판매금액")
])

# 3. 파일로 저장! (데이터가 너무 크면 메모리가 터질 수 있으니 sink_parquet를 추천해)
try:
    # 스트리밍 방식으로 바로 저장 (메모리 절약)
    pos_processed_lazy.sink_parquet("B2_POS_SALE.parquet")
    print("저장 성공! 🎉")
except AttributeError:
    # 혹시 sink_parquet를 지원하지 않는 버전이라면 기존 방식 사용
    pos_processed_lazy.collect().write_parquet("B2_POS_SALE.parquet")
    print("저장 성공! 🎉")

저장 성공! 🎉


In [18]:
# 딱 한 번만 고생해서 변환하자!
# 엑셀을 읽어서 바로 Parquet으로 저장
pl.read_excel(B4_PATH).write_parquet("B4_food_item_data.parquet")

# 이제부터는 엑셀 대신 Parquet을 Lazy하게 사용!
b4_lazy = pl.scan_parquet("B4_food_item_data.parquet")

# 이렇게 하면 b2_lazy와 b4_lazy 모두 '예약' 상태라 안전해!

In [19]:
# 1. 경로 설정
B2_PATH = '/Users/hajiyoon/dataset_seveneleven/B2_POS_SALE.parquet'
B4_PATH = '/Users/hajiyoon/dataset_seveneleven/B4_food_item_data.parquet'


# 2. 데이터 스캔 (Lazy 모드 시작)
b2_lazy = pl.scan_parquet(B2_PATH)
b4_lazy = pl.scan_parquet(B4_PATH)

In [15]:
# 데이터 경로 설정
B2_PATH = '/Users/hajiyoon/dataset_seveneleven/B2_POS_SALE.parquet'
B4_PATH = '/Users/hajiyoon/dataset_seveneleven/B4_food_item_data.xlsx'

# 1. 데이터 로드
def load_b2(path):
    return pl.scan_parquet(path)

b2_lazy = load_b2(B2_PATH)
b4_lazy = pl.read_excel(B4_PATH)


In [4]:
# 2. 데이터 구조 및 샘플 확인
print("\n--- B2 데이터 샘플 (상위 5개 행) ---")
display(b2_lazy.head(5).collect())


--- B2 데이터 샘플 (상위 5개 행) ---


판매일자,판매시간,점포코드,POS번호,거래번호,상품코드,판매수량,판매금액
str,str,str,str,str,str,i64,i64
"""20250523""","""084531""","""64139""","""02""","""26651""","""314952""",1,500
"""20250523""","""105724""","""52657""","""01""","""48727""","""314954""",10,45000
"""20250523""","""105724""","""52657""","""01""","""48727""","""314952""",10,5000
"""20250523""","""084531""","""64139""","""02""","""26651""","""314983""",1,4500
"""20250523""","""083823""","""50134""","""01""","""67971""","""314988""",1,4500


# 1. 데이터 스캐닝 및 기초 프로파일링

- 목적: 대용량 POS 데이터의 물리적 크기를 확인하고, 스키마 및 결측치 구조를 파악하여 데이터의 정합성을 검증한다.
- 데이터 : 전처리한 pos 데이터 

In [ ]:
# 2. 총 데이터 건수 확인
total_rows = b2_lazy.select(pl.len()).collect().item()
print(f"총 데이터 건수: {total_rows}건")

# 3. 데이터 스키마 확인
print("\n데이터 스키마:")
print(b2_lazy.schema)

# 4. 컬럼별 결측치(Null) 개수 집계
null_counts_df = b2_lazy.select(pl.all().null_count()).collect()
print("\n컬럼별 결측치 수:")
print(null_counts_df)



총 데이터 건수: 49758183건

데이터 스키마:
Schema({'판매일자': String, '판매시간': String, '점포코드': String, 'POS번호': String, '거래번호': String, '상품코드': String, '판매수량': Int64, '판매금액': Int64})


/var/folders/y9/_xbnh5z57m96hljq1krz11tr0000gn/T/ipykernel_60725/3467494375.py:7: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(b2_lazy.schema)



컬럼별 결측치 수:
shape: (1, 8)
┌──────────┬──────────┬──────────┬─────────┬──────────┬──────────┬──────────┬──────────┐
│ 판매일자 ┆ 판매시간 ┆ 점포코드 ┆ POS번호 ┆ 거래번호 ┆ 상품코드 ┆ 판매수량 ┆ 판매금액 │
│ ---      ┆ ---      ┆ ---      ┆ ---     ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ u32      ┆ u32      ┆ u32      ┆ u32     ┆ u32      ┆ u32      ┆ u32      ┆ u32      │
╞══════════╪══════════╪══════════╪═════════╪══════════╪══════════╪══════════╪══════════╡
│ 0        ┆ 0        ┆ 0        ┆ 0       ┆ 0        ┆ 0        ┆ 0        ┆ 0        │
└──────────┴──────────┴──────────┴─────────┴──────────┴──────────┴──────────┴──────────┘

수치형 데이터 요약 통계:
shape: (9, 3)
┌────────────┬──────────┬─────────────┐
│ statistic  ┆ 거래번호 ┆ 판매수량    │
│ ---        ┆ ---      ┆ ---         │
│ str        ┆ str      ┆ f64         │
╞════════════╪══════════╪═════════════╡
│ count      ┆ 49758183 ┆ 4.9758183e7 │
│ null_count ┆ 0        ┆ 0.0         │
│ mean       ┆ null     ┆ 1.317539    │
│ std        ┆ null     ┆ 2.991054    │

In [14]:
# 5. 수치형 데이터(판매수량, 판매금액) 기초 통계량 확인
# 기술 통계량 산출을 위해 대상 컬럼만 선택 후 collect() 및 describe() 적용
summary_stats = b2_lazy.select(["판매금액", "판매수량"]).collect().describe()
print("\n수치형 데이터 요약 통계:")
print(summary_stats)


수치형 데이터 요약 통계:
shape: (9, 3)
┌────────────┬───────────────┬─────────────┐
│ statistic  ┆ 판매금액      ┆ 판매수량    │
│ ---        ┆ ---           ┆ ---         │
│ str        ┆ f64           ┆ f64         │
╞════════════╪═══════════════╪═════════════╡
│ count      ┆ 4.9758183e7   ┆ 4.9758183e7 │
│ null_count ┆ 0.0           ┆ 0.0         │
│ mean       ┆ 3265.605813   ┆ 1.317539    │
│ std        ┆ 368765.110163 ┆ 2.991054    │
│ min        ┆ -1.3000e9     ┆ -2000.0     │
│ 25%        ┆ 1500.0        ┆ 1.0         │
│ 50%        ┆ 2300.0        ┆ 1.0         │
│ 75%        ┆ 4500.0        ┆ 1.0         │
│ max        ┆ 1.3000e9      ┆ 2000.0      │
└────────────┴───────────────┴─────────────┘


In [15]:
print("\n--- [검토 2] 판매수량 및 판매금액 이상치 점검 (음수 매출 확인) ---")
numeric_stats = b2_lazy.select([
    pl.col("판매수량").min().alias("수량_최소"),
    pl.col("판매수량").max().alias("수량_최대"),
    (pl.col("판매수량") < 0).sum().alias("수량_0미만_건수"),
    pl.col("판매금액").min().alias("금액_최소"),
    pl.col("판매금액").max().alias("금액_최대"),
    (pl.col("판매금액") < 0).sum().alias("금액_0미만_건수")
]).collect()
display(numeric_stats)


--- [검토 2] 판매수량 및 판매금액 이상치 점검 (음수 매출 확인) ---


수량_최소,수량_최대,수량_0미만_건수,금액_최소,금액_최대,금액_0미만_건수
i64,i64,u32,i64,i64,u32
-2000,2000,1056896,-1300000000,1300000000,1030054


In [16]:

# 가설 1 검증: 극단치 대칭 발생 영수증 추적
# 절댓값이 비정상적으로 큰 거래를 필터링하여 동일 점포 및 시간대에 양수/음수 거래가 발생했는지 확인한다.
hypothesis_1_verify = b2_lazy.filter(
    (pl.col("판매수량").abs() >= 1000) | (pl.col("판매금액").abs() >= 1000000)
).select(
    ["판매일자", "판매시간", "점포코드", "거래번호", "상품코드", "판매수량", "판매금액"]
).sort(["점포코드", "판매일자", "판매시간"]).collect()

print("--- [검증 1] 극단치 즉시 취소 내역 ---")
display(hypothesis_1_verify.head(10))

# 가설 2 검증: 수량과 금액 부호 불일치 집단(할인/프로모션) 격리
# 수량은 0 이상인데 금액만 음수인 데이터를 분리하여, 특정 할인용 '상품코드'에 집중되어 있는지 확인한다.
hypothesis_2_verify = b2_lazy.filter(
    (pl.col("판매수량") >= 0) & (pl.col("판매금액") < 0)
).group_by("상품코드").agg(
    pl.len().alias("발생_건수"),
    pl.col("판매금액").mean().alias("평균_할인액")
).sort("발생_건수", descending=True).collect()

print("\n--- [검증 2] 할인/프로모션 추정 상품코드 ---")
display(hypothesis_2_verify.head(10))

# 가설 3 검증: 정상 판매 후 취소에 걸린 리드타임(Lead Time) 분석
# 극단치와 프로모션을 제외한 일반 음수 거래의 비중을 확인한다.
# 이를 통해 향후 EDA 2단계(상품별 누적 매출 집계)에서 제외할 타겟을 확정한다.
hypothesis_3_verify = b2_lazy.filter(
    (pl.col("판매수량") < 0) & 
    (pl.col("판매수량") >= -50)  # 소량 환불건 한정
).select(pl.len().alias("일반_환불_건수")).collect()

print("\n--- [검증 3] 일반 환불 건수 규모 ---")
display(hypothesis_3_verify)

--- [검증 1] 극단치 즉시 취소 내역 ---


판매일자,판매시간,점포코드,거래번호,상품코드,판매수량,판매금액
str,str,str,str,str,i64,i64
"""20250113""","""113109""","""10043""","""79059""","""107872""",200,1960000
"""20250113""","""113109""","""10043""","""79059""","""315183""",220,1980000
"""20250113""","""113250""","""10043""","""79060""","""107872""",-200,-1960000
"""20250113""","""113250""","""10043""","""79060""","""315183""",-220,-1980000
"""20250113""","""113435""","""10043""","""79064""","""107872""",200,1960000
"""20250113""","""113435""","""10043""","""79064""","""315384""",220,2156000
"""20250402""","""154949""","""10043""","""66025""","""315384""",200,1960000
"""20250624""","""165238""","""10043""","""77306""","""315384""",300,2940000
"""20250610""","""092301""","""10177""","""37646""","""315274""",400,1920000



--- [검증 2] 할인/프로모션 추정 상품코드 ---


상품코드,발생_건수,평균_할인액
str,u32,f64
"""470273""",292,-853.523973
"""470364""",129,-335.829457
"""104848""",2,-5.0



--- [검증 3] 일반 환불 건수 규모 ---


일반_환불_건수
u32
1054923


1. 데이터 정제 (Data Cleansing) 전략
본 프로젝트는 '일반 소비자의 취향(트렌드)'을 분석하는 것이 목적이므로,
분석의 정확도를 떨어뜨리는 특수 목적의 구매나 시스템 노이즈를 다음과 같은 기준으로 정제합니다.

① 극단치(Outlier) 제거: B2B 및 입력 오류 배제

- 기준: 1회 결제 시 판매수량의 절댓값이 50개를 초과하는 데이터 제거 (abs(판매수량) <= 50)
- 이유: 한 번에 100개 이상 구매/취소하는 패턴은 일반 소비자가 아닌 학교/기관의 행사용 대량 납품이거나 포스기 조작 실수일 확률이 99%입니다. 이를 포함하면 특정 상품의 성과가 비정상적으로 부풀려져 트렌드 분석이 왜곡됩니다.

② 반품(Negative) 데이터 유지: 순매출(Net Sales) 기반 평가

- 기준: 수량이 음수(-50 ~ -1)인 정상 반품 트랜잭션은 삭제하지 않고 데이터셋에 포함합니다.
- 이유: 반품 데이터를 제거하고 양수(+) 매출만 집계할 경우, '많이 팔렸지만 그만큼 환불도 많이 된 불량 상품'이 히트 상품으로 둔갑하는 오류(Data Leakage)가 발생합니다.

2. 핵심 성과 지표 (Target Metric) 정의
상품이 시장에서 '성공'했는지를 평가하기 위해, 출시 시점이 서로 다른 상품들을 동일한 출발선에서 평가합니다.

- 지표명: 출시 초기 4주(28일) 누적 순매출액
- 산출 방식:
1.  각 상품별 최초 판매일을 출시일(Launch Date)로 정의합니다.
2.  결제일과 출시일의 차이를 계산하여 경과일(Days After)을 도출합니다.
3.  출시 후 1주/2주/4주 이내에 발생한 트랜잭션의 판매금액을 단순 합산(.sum())합니다.
4.  (반품 데이터가 섞여 있으므로 합산 시 자동으로 순매출액이 도출됩니다.)

In [5]:
print("데이터 정제 및 초기 4주 누적 순매출 집계를 시작합니다...")

# ==========================================
# [Step 1] 데이터 정제 (Data Cleansing)
# ==========================================
# B2B 대량 구매/오입력(50개 초과) 배제 및 순수 결제/반품만 남김
b2_clean_lazy = b2_lazy.filter(
    (pl.col("판매수량").abs() <= 50) & 
    (pl.col("판매수량") != 0)
)

# ==========================================
# [Step 2] 날짜 변환 및 출시일(Launch Date) 정의
# ==========================================
# 문자열 날짜를 Date 타입으로 변환
b2_with_date = b2_clean_lazy.with_columns(
    pl.col('판매일자').str.to_date('%Y%m%d').alias('sale_dt')
)

# 상품별로 가장 빠른 결제일을 '출시일'로 지정
launch_dates = b2_with_date.group_by('상품코드').agg(
    pl.col('sale_dt').min().alias('launch_dt')
)

# ==========================================
# [Step 3] 경과일(Days After) 계산 및 타겟 변수 집계
# ==========================================
# 원본 데이터에 출시일을 붙이고 결제 시점의 '경과일' 계산
sales_enriched = b2_with_date.join(launch_dates, on='상품코드')
sales_enriched = sales_enriched.with_columns(
    (pl.col('sale_dt') - pl.col('launch_dt')).dt.total_days().alias('days_after')
)

# 출시 후 1주(7일), 2주(14일), 4주(28일) 이내의 금액 합산 (순매출)
cum_sales_lazy = sales_enriched.group_by('상품코드').agg([
    pl.col('판매금액').filter(pl.col('days_after') <= 7).sum().alias('cum_1w'),
    pl.col('판매금액').filter(pl.col('days_after') <= 14).sum().alias('cum_2w'),
    pl.col('판매금액').filter(pl.col('days_after') <= 28).sum().alias('cum_4w')
])

# ==========================================
# [Step 4] 연산 실행(Collect) 및 결과 확인
# ==========================================
cum_sales = cum_sales_lazy.collect()

print(f"✅ 분석 대상 고유 상품 수: {len(cum_sales):,} 개")
print("\n--- [결과] 4주 누적 순매출 Top 5 상품 ---")
print(cum_sales.sort('cum_4w', descending=True).head(5))

# (선택) 다음 분석을 위해 결과를 파일로 저장
# cum_sales.write_parquet("target_variable_cum_sales.parquet")

데이터 정제 및 초기 4주 누적 순매출 집계를 시작합니다...
✅ 분석 대상 고유 상품 수: 12,598 개

--- [결과] 4주 누적 순매출 Top 5 상품 ---
shape: (5, 4)
┌──────────┬──────────┬───────────┬───────────┐
│ 상품코드 ┆ cum_1w   ┆ cum_2w    ┆ cum_4w    │
│ ---      ┆ ---      ┆ ---       ┆ ---       │
│ str      ┆ i64      ┆ i64       ┆ i64       │
╞══════════╪══════════╪═══════════╪═══════════╡
│ 314954   ┆ 62361000 ┆ 117616500 ┆ 219645000 │
│ 314631   ┆ 59319000 ┆ 113575500 ┆ 215248500 │
│ 300112   ┆ 59112000 ┆ 111537000 ┆ 210289500 │
│ 314983   ┆ 54427500 ┆ 104485500 ┆ 193410000 │
│ 315275   ┆ 54014400 ┆ 103123200 ┆ 192600000 │
└──────────┴──────────┴───────────┴───────────┘


In [8]:
import sys
!{sys.executable} -m pip install pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [12]:
import polars as pl

# 1. 상위 20% 성공 기준 금액(커트라인) 계산
# cum_sales는 아까 계산해둔 그 테이블이야!
threshold_80 = cum_sales.select(pl.col("cum_4w").quantile(0.80)).item()

print(f"🏆 [분석 결과] 상위 20% 성공 기준 커트라인")
print(f"👉 4주 누적 순매출액: {threshold_80:,.0f} 원")
print("-" * 50)

# 2. 성공(1) / 실패(0) 라벨링 (Target Variable 생성)
labeled_sales = cum_sales.with_columns(
    pl.when(pl.col("cum_4w") >= threshold_80)
    .then(1)
    .otherwise(0)
    .alias("is_success")
)

# 3. 데이터 분포 확인 (간이 테이블 버전)
summary = labeled_sales.group_by("is_success").agg([
    pl.len().alias("상품_수"),
    pl.col("cum_4w").mean().alias("평균_매출"),
    pl.col("cum_4w").max().alias("최대_매출")
])

print("--- [라벨링 결과 요약] ---")
print(summary)

# 4. 나중에 모델링에 쓸 수 있게 데이터 저장 (차트 안 떠도 저장은 잘 됨!)
labeled_sales.write_parquet("target_variable_labeled.parquet")
print("\n✅ 데이터 저장 완료: target_variable_labeled.parquet")

🏆 [분석 결과] 상위 20% 성공 기준 커트라인
👉 4주 누적 순매출액: 2,039,199 원
--------------------------------------------------
--- [라벨링 결과 요약] ---
shape: (2, 4)
┌────────────┬─────────┬───────────────┬───────────┐
│ is_success ┆ 상품_수 ┆ 평균_매출     ┆ 최대_매출 │
│ ---        ┆ ---     ┆ ---           ┆ ---       │
│ i32        ┆ u32     ┆ f64           ┆ i64       │
╞════════════╪═════════╪═══════════════╪═══════════╡
│ 1          ┆ 2520    ┆ 9.6726e6      ┆ 219645000 │
│ 0          ┆ 10078   ┆ 345054.528478 ┆ 2038025   │
└────────────┴─────────┴───────────────┴───────────┘

✅ 데이터 저장 완료: target_variable_labeled.parquet


출시 후 4주(한 달) 동안 순매출이 2,039,199원을 넘으면 우리는 이 상품을 '성공(Hit)'이라고 정의

# 2. 상품 속성(Attribute) 추출

[수행할 작업: 상품 마스터 데이터 파싱]

- 텍스트 마이닝: 상품명에서 핵심 키워드(예: 마라, 초코, 저칼로리, 한정판)를 정규표현식으로 추출하기.
- 가격 밴드 생성: 판매가를 기준으로 '초저가 / 저가 / 중가 / 고가' 등의 범주 만들기.
- 카테고리 매칭: PB 상품 여부나 특정 브랜드 상품 여부 확인하기.

In [22]:
b4_head = b4_lazy.head(10).collect()
    
print("--- 음식 데이터 상위 10개 행 ---")
print(b4_head)

--- 음식 데이터 상위 10개 행 ---
shape: (10, 5)
┌─────────┬───────────────────────────────┬──────────────┬──────────────┬──────────────┐
│ ITEM_CD ┆ ITEM_NM                       ┆ ITEM_LRDV_NM ┆ ITEM_MDDV_NM ┆ ITEM_SMDV_NM │
│ ---     ┆ ---                           ┆ ---          ┆ ---          ┆ ---          │
│ str     ┆ str                           ┆ str          ┆ str          ┆ str          │
╞═════════╪═══════════════════════════════╪══════════════╪══════════════╪══════════════╡
│ 040481  ┆ CJ)후레쉬젤리루비자몽105g     ┆ 디저트       ┆ 젤리         ┆ 떠먹는젤리   │
│ 040483  ┆ 샤니)소프트밀키케익           ┆ 디저트       ┆ 냉장베이커리 ┆ 롤케익       │
│ 040486  ┆ 제로칼로리젤리(아사히베리)    ┆ 디저트       ┆ 젤리         ┆ 떠먹는젤리   │
│ 040488  ┆ 브)스위트브라우니             ┆ 디저트       ┆ 냉장베이커리 ┆ 냉장과자     │
│ 040491  ┆ 아띠)스위트젤리(밀감)250g     ┆ 디저트       ┆ 젤리         ┆ 떠먹는젤리   │
│ 040493  ┆ Cafe Seven)케익속에리얼딸기   ┆ 디저트       ┆ 냉장베이커리 ┆ 롤케익       │
│ 040496  ┆ 한입경단                      ┆ 디저트       ┆ 떡           ┆ 떡           │
│ 040499  ┆ 모닝후르츠)모듬과일도시

In [23]:
# [수정] B4의 ITEM_CD를 '상품코드'로 미리 이름을 바꿔두기 (Join을 위해)
b4_lazy = b4_lazy.rename({"ITEM_CD": "상품코드"})

# 3. 누적 매출 및 성공 라벨링 로직 (위에서 했던 것과 동일)
# ... (중략: b2_cleaned, launch_dates, cum_sales 계산 과정) ...
# SUCCESS_THRESHOLD = 2039199

# 4. 음식 데이터와 결합 및 속성 추출
final_analysis_lazy = b4_lazy.join(cum_sales, on="상품코드", how="inner").with_columns([
    # 성공 라벨링
    pl.when(pl.col("cum_4w") >= 2039199).then(1).otherwise(0).alias("is_success"),
    
    # 카테고리 정보와 상품명 키워드 동시 활용
    pl.col("ITEM_NM").str.contains("매운|불닭|마라").alias("attr_spicy"),
    pl.col("ITEM_NM").str.contains("제로|0칼로리").alias("attr_zero"),
    
    # 중분류명을 그대로 속성으로 활용 (예: 젤리, 떡 등)
    pl.col("ITEM_MDDV_NM").alias("category_md")
])

final_df = final_analysis_lazy.collect()

# 5. 결과 확인: 어떤 '중분류'가 가장 성공률이 높을까?
category_success = final_df.group_by("category_md").agg([
    pl.col("is_success").mean().alias("성공률"),
    pl.len().alias("상품수")
]).sort("성공률", descending=True)

print("--- [카테고리(중분류)별 성공률 순위] ---")
print(category_success.head(10))

--- [카테고리(중분류)별 성공률 순위] ---
shape: (10, 3)
┌─────────────┬──────────┬────────┐
│ category_md ┆ 성공률   ┆ 상품수 │
│ ---         ┆ ---      ┆ ---    │
│ str         ┆ f64      ┆ u32    │
╞═════════════╪══════════╪════════╡
│ 삼각김밥    ┆ 0.74     ┆ 50     │
│ 스포츠음료  ┆ 0.666667 ┆ 33     │
│ 냉장피자    ┆ 0.666667 ┆ 24     │
│ 요구르트    ┆ 0.65625  ┆ 32     │
│ 샌드위치    ┆ 0.583333 ┆ 60     │
│ 차음료      ┆ 0.583333 ┆ 96     │
│ 김밥        ┆ 0.576923 ┆ 52     │
│ 프로틴 음료 ┆ 0.576923 ┆ 26     │
│ 도시락      ┆ 0.52439  ┆ 82     │
│ 커피음료    ┆ 0.521739 ┆ 92     │
└─────────────┴──────────┴────────┘


요약
- 삼각김밥 등 간편식과 스포츠/프로틴 음료 등 기능성 카테고리의 신상품 성공률이 두드러지게 높게 나타났다.

- 커피 및 차음료는 신제품 출시는 가장 활발하나, 경쟁이 심화되어 평균 성공률은 상대적으로 낮다.

- 향후 분석에서는 이러한 카테고리 노드에 제로, 단백질, 매운맛 등의 세부 속성 노드를 결합하여, 성공 확률을 극대화하는 특정 조합(성공 방정식)을 HIN 기반으로 탐색한다.